# Transformer architecture

This is a diagram you can find all over the internet, illustrating the transformer architecture. Our job today is to understand the diagram and what all the parts do.

<img src="img/attention_research_1.png" width="500">

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from lark import Lark

## Task to study

We'll do it in the context of a problem to solve: identifying and then generating valid sentences in a very simple language.

Below is the entire language, expressed as a [BNF grammar](https://en.wikipedia.org/wiki/Backus%E2%80%93Naur_form):

In [30]:
parser = Lark("""
    sentence:    subj    intrans
            |    subj    trans   obj
            | pp subj    intrans
            |    subj    intrans     pp
            |    subj pp intrans
            | pp subj    trans   obj
            |    subj    trans   obj pp
            |    subj pp trans   obj

    subj:      "THE" entity |      "A" entity_cons |      "AN" entity_vowel
    obj:       "THE" entity |      "A" entity_cons |      "AN" entity_vowel
    pp:   prep "THE" place  | prep "A" place_cons  | prep "AN" place_vowel

    entity:       entity_cons | entity_vowel
    place:        place_cons | place_vowel

    entity_cons:  "CAT" | "MOUSE" | "ROBOT"
    entity_vowel: "ALLIGATOR" | "ANT"
    place_cons:   "HOUSE" | "YARD" | "STREET"
    place_vowel:  "AIRPORT" | "OCEAN"

    prep:         "IN" | "UNDER" | "ABOVE"
    intrans:      "SMILES" | "BURPS" | "SLEEPS" | "EXPLODES"
    trans:        "CHASES" | "EATS" | "LOVES" | "HATES"

    %ignore " "
""", start="sentence", parser="lalr", keep_all_tokens=True)

This parser does the task algorithmically (`LALR(1)`). We'll want to train a model to do the same task.

In [38]:
print(parser.parse("THE CAT SMILES").pretty(indent_str="    ").replace("\t", ": "))

sentence
    subj
        THE
        entity
            entity_cons: CAT
    intrans: SMILES


In [39]:
print(parser.parse("THE CAT EATS A MOUSE IN THE STREET").pretty(indent_str="    ").replace("\t", ": "))

sentence
    subj
        THE
        entity
            entity_cons: CAT
    trans: EATS
    obj
        A
        entity_cons: MOUSE
    pp
        prep: IN
        THE
        place
            place_cons: STREET


Data will be given to the model as lists of 8 tokens, using `<BLANK>` as filler at the end.

This `is_valid` function is a drop-in replacement for what we'll want our first model to do.

In [53]:
BLANK = "<BLANK>"

SEQUENCE_LENGTH = 8

def is_valid(tokens):
    # length must be SEQUENCE_LENGTH
    if len(tokens) != SEQUENCE_LENGTH:
        return False

    # all tokens after the first BLANK must be BLANK, allowing for no BLANKs by adding one
    if not all(x == BLANK for x in tokens[(tokens + [BLANK]).index(BLANK):]):
        return False

    try:
        # try to parse the string without BLANKs
        parser.parse(" ".join([x for x in tokens if x != BLANK]))
        return True
    except Exception:
        return False

In [54]:
is_valid(["THE", "CAT", "SMILES", BLANK, BLANK, BLANK, BLANK, BLANK])

True

In [55]:
is_valid(["THE", "CAT", "EATS", "A", "MOUSE", "IN", "THE", "STREET"])

True

In [56]:
is_valid(["THE", "CAT", "EATS", BLANK, BLANK, BLANK, BLANK, BLANK])

False

## Dataset

Random sentences labeled by validity, in which 50% are `ok` and 50% are `bad`.

In [49]:
train_df    = pd.read_csv("data/train.csv",    nrows=20000)
validate_df = pd.read_csv("data/validate.csv", nrows=20000)
test_df     = pd.read_csv("data/test.csv",     nrows=20000)

In [50]:
train_df

,t0,t1,t2,t3,t4,t5,t6,t7,label
0,ABOVE,A,STREET,THE,MOUSE,HATES,A,CAT,ok
1,IN,A,HOUSE,AN,ANT,CHASES,A,MOUSE,ok
2,IN,THE,STREET,THE,ANT,CHASES,THE,ANT,ok
3,IN,MOUSE,YARD,A,ROBOT,A,LOVES,A,bad
4,IN,A,YARD,ANT,ALLIGATOR,CHASES,AN,AN,bad
...,...,...,...,...,...,...,...,...,...
19995,A,ROBOT,CHASES,THE,ANT,UNDER,AN,AIRPORT,ok
19996,UNDER,AN,OCEAN,AN,ANT,HATES,THE,ALLIGATOR,ok
19997,AIRPORT,THE,AIRPORT,CAT,ALLIGATOR,LOVES,A,MOUSE,bad
19998,THE,ALLIGATOR,LOVES,AN,ANT,UNDER,A,YARD,ok


One-hot encoding for the 25 tokens:

In [82]:
token_index = {
    x: i for i, x in enumerate([
        "THE", "A", "AN",
        "CAT", "MOUSE", "ROBOT", "ALLIGATOR", "ANT",
        "HOUSE", "YARD", "STREET", "AIRPORT", "OCEAN",
        "IN", "UNDER", "ABOVE",
        "SMILES", "BURPS", "SLEEPS", "EXPLODES",
        "CHASES", "EATS", "LOVES", "HATES",
        BLANK
    ])
}
VOCABULARY_SIZE = len(token_index)

df = train_df

x = torch.zeros((len(df), SEQUENCE_LENGTH, VOCABULARY_SIZE))
token_columns = [f"t{i}" for i in range(SEQUENCE_LENGTH)]
indexes = df[token_columns].apply(lambda col: col.map(token_index).values)

In [83]:
for i, colname in enumerate(token_columns):
    x[:, i, indexes[colname]] = 1

In [84]:
x

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        ...,

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1., 